## Background Knowledge
- `n_embd`: number of dimensions of embedding vector of a token.

## Block
In paper, we can see that the author defines block as a sequence of layers (right side of the figure).
For the first step, we define block having only masked multi-head and feedforward layers

<img src="./assets/model-architecture.png" style="height: 400px">

The loss is actually worse than previous implementation `step 4800: train loss 2.3217, val loss 2.3387`. Because now we have a deep neural net. So we need following optimizations.

## Optimizations
### Residual NN
- Background
    - During back propagation, addition spreads gradients equivalent to both branches.
### Update Feedforward
- Add another linear later, now the structure conform with the paper description.
- The d_ff should be 4 * d_model which is input and output dimension.

### [Layer Norm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)
Layer norm vs. Batch norm
- LN normalizing across features for a single sample, whereas BN normalizes across the batch for each feaure.
- We don't need buffer part (running_mean and running_var) since we dont normalize across examples.
- Slighlty deviate from the paper, the modern convetion is that we apply norm before any transformation
- Add layer norm in each layer of a block and after the blocks
## Refs
- [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)
- [Dropout paper](https://www.cs.toronto.edu/~rsalakhu/papers/srivastava14a.pdf)

In [34]:
## Batchnorm to Layer norm this is just for illustration
## The implementation is pretty much what the pytorch LayerNorm do.
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    # dim = 0 # batch norm
    dim = 1 # layer norm
    xmean = x.mean(dim, keepdim=True) # batch mean
    xvar = x.var(dim, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

In [35]:
feat_dim = 100
batch_size = 32
model = BatchNorm1d(feat_dim)
x = torch.randn(batch_size, feat_dim)
x = model(x)

print("Batch norm: ", x[:, 0].mean(), x[:, 0].std()) # Batch norm normalizes over batch (over column, dim = 0, collapse rows)
print("Layer norm", x[0, :].mean(), x[0, :].std()) # Layer norm normalizes over features (over row, dim = 1, collapse collumns)

Batch norm:  tensor(-0.1942) tensor(0.9110)
Layer norm tensor(4.7684e-09) tensor(1.0000)


In [36]:
batch, sentence_length, embedding_dim = 2, 2, 4
embedding = torch.randn(batch, sentence_length, embedding_dim)
layer_norm = nn.LayerNorm(embedding_dim)
# Activate module
layer_norm(embedding)

tensor([[[-1.3450,  0.7595,  1.1421, -0.5566],
         [-0.6057,  1.4887,  0.2749, -1.1579]],

        [[-0.4666,  1.0690,  0.8074, -1.4098],
         [-0.0575,  1.5630, -1.2113, -0.2942]]],
       grad_fn=<NativeLayerNormBackward0>)

In [37]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('./assets/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # torch.nn.Module.register_buffer is a method used to register a tensor as a "buffer" in a PyTorch module. Buffers are part of the module's state but are not considered learnable parameters by the optimizer. 
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # Ensure the result dimension is (B, T, n_embd)
        self.proj = nn.Linear(head_size * num_heads, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.LayerNorm(n_embd) 
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        

    def forward(self, x):
        # Residual connection
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # i.e. 4 heads of 8-dimensional self-attention (4 * 8 -> 32 = n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head = 4),
            Block(n_embd, n_head = 4),
            Block(n_embd, n_head = 4),
        )
        
        # language model head
        self.lm_head = nn.Linear(n_embd, vocab_size) 

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C) C: embed dimension
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        # decode
        logits = self.lm_head(x) # (B, T, vocab_size) 
        

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        print(idx.shape)
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 5.0740, val loss 5.0610
step 300: train loss 2.4828, val loss 2.4856
step 600: train loss 2.3524, val loss 2.3597
step 900: train loss 2.2647, val loss 2.2811
step 1200: train loss 2.1982, val loss 2.2387
step 1500: train loss 2.1583, val loss 2.2040
step 1800: train loss 2.1308, val loss 2.1774
step 2100: train loss 2.0980, val loss 2.1534
step 2400: train loss 2.0860, val loss 2.1419
step 2700: train loss 2.0773, val loss 2.1328
step 3000: train loss 2.0489, val loss 2.1029
step 3300: train loss 2.0384, val loss 2.1021
step 3600: train loss 2.0185, val loss 2.0883
step 3900: train loss 2.0255, val loss 2.0945
step 4200: train loss 2.0100, val loss 2.0814
step 4500: train loss 2.0004, val loss 2.0837
step 4800: train loss 1.9618, val loss 2.0652


In [39]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

torch.Size([1, 1])

Fit, Lord, A my kendrifent ait colinfy'd ceeem.

ARGREOBUMONTE Bet weckeen eet suffen dare.

CAPETES:
By tood it reach chear herns pret her her is never:
My fau you they bustom
EvEFORK:
MOORD KINGES:
Why mistake perk?, betwe perce of 'She supfees,
Lot did not
I leenten's our thy, come fries up did.

KING ZARD:
Yourbles seire it and his our somebuty,
Ypeing thou know
The swilk
Rome, upcy ember stile;
The nome, hen boogethat this in shall a the be thy 'to
Thing bedeme.
O,
'Tath me I neep. Whath th


## Log Performance 
Implement Block
- `step 4800: train loss 2.3217, val loss 2.3387` <br/>

Implement FeedForward with residual connectionss
- `step 4800: train loss 1.9847, val loss 2.0776` <br/>
- Now the implementation is better than just implementing 1 layer of multihead and rudiment feedforward which has no residual connection.
- Now the val loss - train loss is larger and larger, so we can see the NN is complicated enough to have overfitting.

Implement Layer norm
- `step 4800: train loss 1.9618, val loss 2.0652`